# Step 19.3 — Training Stability

本节覆盖 initialization、gradient norms、optional gradient clipping、NaN/divergence detection，承接19.2的训练/验证循环与loss参数版本边界，不进入资源控制或后续训练实验。

19.2提交 `8a83021c12dd8f5561b562d4156238cefd507a84`已独立核心静态验收PASS，累计修正1/2。这是材料验收，不是学员掌握或运行反馈。

本母版所有代码单元均未执行、无输出。本节初始生成0/2；完成即停在19.3。

## 0. 自包含与依赖恢复

本Notebook含必要导入、候选配置、可选获取定义、本地语料读取、90/10 split、sampler、完整MiniGPT、scalar loss、validation helper、Adam与训练步骤定义。缓存默认在仓库根目录 `data/tinyshakespeare/input.txt`；获取开关默认关闭，缓存缺失明确停止。工作副本从头按顺序恢复，不依赖旧kernel。

In [ ]:
import math
import random
from pathlib import Path
from urllib.request import urlopen
import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim
from mlx.utils import tree_flatten, tree_map

baseline_candidate = {'model_width':64,'layers':2,'heads':4,'block_size':64,'batch_size':4,'learning_rate':3e-4}
seeds = {'model':2028,'train':2026,'validation':2027}
def make_model_kwargs(c,vocab_size):
    return {'vocab_size':vocab_size,'dims':c['model_width'],'num_heads':c['heads'],'num_blocks':c['layers'],'max_seq_len':c['block_size']}
def make_optimizer_kwargs(c):
    return {'learning_rate':c['learning_rate'],'betas':[0.9,0.999],'eps':1e-8,'bias_correction':True} # explicit course choice; MLX default False

## 1. 初始化与候选配置

Initialization决定训练起点，不等于已验证的稳定配置。模型沿用MLX Linear的既有初始化；本节只在构造前设置MLX seed，不替换初始化策略。固定seed有助于复现同一环境下的随机流程，不保证跨版本/设备逐bit一致。

Global gradient norm把所有可训练梯度叶子的元素视为一个长向量。它是非负标量，不是loss，也不是Adam更新后参数移动的距离。

## 2. 数据恢复与边界（未来练习）

可选获取入口默认关闭；本地读取不隐式下载。先恢复字符表和token-ID映射，再做连续90/10 split；两侧都至少需要block_size+1个token。

In [ ]:
SOURCE_URL='https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
def find_project_root(start):
    current=Path(start).resolve()
    for p in [current]+list(current.parents):
        if (p/'pyproject.toml').is_file() and (p/'notebooks'/'82_step19_1_hyperparameter_baseline.ipynb').is_file(): return p
    raise FileNotFoundError('Set PROJECT_ROOT to repository root.')
def acquire_tiny_shakespeare(path,source_url=SOURCE_URL):
    path=Path(path)
    if path.is_file(): return path
    path.parent.mkdir(parents=True,exist_ok=True)
    with urlopen(source_url,timeout=30) as response: raw=response.read()
    tmp=path.with_suffix('.tmp'); tmp.write_bytes(raw); tmp.replace(path); return path
def load_cached_character_corpus(path):
    path=Path(path)
    if not path.is_file(): raise FileNotFoundError(f'Cache missing: {path.resolve()}; correct path or explicitly acquire it.')
    text=path.read_text(encoding='utf-8'); chars=sorted(set(text)); stoi={ch:i for i,ch in enumerate(chars)}; itos={i:ch for i,ch in enumerate(chars)}
    return text,chars,stoi,itos,mx.array([stoi[ch] for ch in text],dtype=mx.int32)
def split_train_validation(stream,fraction=0.90):
    if len(stream.shape)!=1 or stream.shape[0]<2 or not 0<fraction<1: raise ValueError('Invalid stream/split')
    i=int(stream.shape[0]*fraction)
    if i<=0 or i>=stream.shape[0]: raise ValueError('Empty split')
    return stream[:i],stream[i:],i
# Future work-copy bootstrap; acquisition remains explicitly disabled.
PROJECT_ROOT=find_project_root(Path.cwd())
CACHE_PATH=PROJECT_ROOT/'data'/'tinyshakespeare'/'input.txt'
ACQUIRE_MISSING_DATA=False
if ACQUIRE_MISSING_DATA: acquire_tiny_shakespeare(CACHE_PATH)
text,characters,stoi,itos,token_stream=load_cached_character_corpus(CACHE_PATH)
train_tokens,val_tokens,split_index=split_train_validation(token_stream,0.90)
vocab_size=len(characters)
for name,split in (('train',train_tokens),('validation',val_tokens)):
    if split.shape[0]<baseline_candidate['block_size']+1: raise ValueError(f'{name} split needs block_size+1 tokens')
mx.eval(train_tokens,val_tokens)

## 3. Sampler、完整MiniGPT与loss

Sampler只从传入的split内部取T+1个连续token，返回X/Y各(B,T)，Y已经shift。模型类沿用已验收的完整15.4实现，不用toy SharedLogitModel代替。Loss对raw logits(B,T,V)与已shift targets(B,T)按相同顺序flatten、cross-entropy后mean，返回标量MLX数组。

In [ ]:
def get_batch(data, batch_size: int, block_size: int, rng: random.Random):
    """Sample shifted (X, Y) from one already-selected integer token split."""
    if len(data.shape) != 1:
        raise ValueError('data must be one token stream with shape (N_split,)')
    if type(batch_size) is not int or batch_size < 1:
        raise ValueError('batch_size must be a positive Python int')
    if type(block_size) is not int or block_size < 1:
        raise ValueError('block_size must be a positive Python int')

    N_split = data.shape[0]
    if N_split < block_size + 1:
        raise ValueError('the selected split needs at least block_size + 1 tokens')

    starts = [
        rng.randrange(0, N_split - block_size)
        for _ in range(batch_size)
    ]

    xs = []
    ys = []
    for start in starts:
        chunk = data[start : start + block_size + 1]  # (T+1,)
        xs.append(chunk[:-1])                         # (T,)
        ys.append(chunk[1:])                          # (T,)

    X = mx.stack(xs, axis=0)  # (B,T)
    Y = mx.stack(ys, axis=0)  # (B,T)
    return X, Y, starts


# Definition only. No batch is sampled in this cell.


In [ ]:
import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        dims: int,
        num_heads: int,
        num_blocks: int,
        max_seq_len: int,
    ):
        super().__init__()

        if vocab_size <= 0:
            raise ValueError('vocab_size must be positive')
        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if num_blocks <= 0:
            raise ValueError('num_blocks must be positive')
        if max_seq_len <= 0:
            raise ValueError('max_seq_len must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        # Python metadata: these are not trainable mx.array parameters.
        self.vocab_size = vocab_size
        self.dims = dims
        self.max_seq_len = max_seq_len

        # Learned lookup tables.
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            dims=dims,
        )
        self.position_embedding = nn.Embedding(
            num_embeddings=max_seq_len,
            dims=dims,
        )

        # A list of distinct Module instances; MLX recursively tracks them.
        self.blocks = [
            TransformerBlock(
                dims=dims,
                num_heads=num_heads,
            )
            for _ in range(num_blocks)
        ]

        self.final_norm = nn.RMSNorm(dims=dims)
        self.lm_head = nn.Linear(
            input_dims=dims,
            output_dims=vocab_size,
            bias=False,
        )

    def __call__(self, token_ids: mx.array) -> mx.array:
        if len(token_ids.shape) != 2:
            raise ValueError('token_ids must have shape (B,T)')

        B, T = token_ids.shape

        if T < 1:
            raise ValueError('sequence length T must be at least 1')
        if T > self.max_seq_len:
            raise ValueError(
                f'sequence length {T} exceeds max_seq_len {self.max_seq_len}'
            )

        # Forward-local structural indices: not Module parameters.
        position_ids = mx.arange(T)  # (T,)

        token_features = self.token_embedding(token_ids)        # (B,T,C)
        position_features = self.position_embedding(position_ids)  # (T,C)
        x = token_features + position_features                  # (B,T,C)

        for block in self.blocks:
            x = block(x)  # invariant: (B,T,C)

        x = self.final_norm(x)  # (B,T,C)
        logits = self.lm_head(x)  # (B,T,V)

        expected_shape = (B, T, self.vocab_size)
        if logits.shape != expected_shape:
            raise RuntimeError(
                f'logits shape {logits.shape} != expected {expected_shape}'
            )

        return logits


# Definition only. No MiniGPT forward is executed in this master notebook.


In [ ]:
def loss_fn(model, X, Y):
    # X, Y: (B, T); Y is already shifted by get_batch.
    logits = model(X)  # (B, T, V), raw unnormalized logits
    B, T, V = logits.shape
    flat_logits = mx.reshape(logits, (B * T, V))  # (B*T, V)
    flat_targets = mx.reshape(Y, (B * T,))        # (B*T,)
    token_losses = nn.losses.cross_entropy(
        flat_logits, flat_targets, axis=-1, reduction="none"
    )  # (B*T,)
    return mx.mean(token_losses)  # scalar MLX array, shape ()


## 4. Validation helper与模式

验证直接调用loss_fn，不使用nn.value_and_grad，也不接收optimizer；因此验证没有更新路径。`model.eval()`切换模块模式但不关闭梯度；当前MiniGPT没有Dropout、BatchNorm或依赖`self.training`的前向分支。每批先mx.eval(loss)，再读取标量；完成后恢复原模式。

In [ ]:
def estimate_validation_loss(model,val_tokens,batch_size,block_size,num_batches,seed):
    if type(num_batches) is not int or num_batches<1: raise ValueError('num_batches must be positive')
    rng=random.Random(seed); was_training=model.training; model.eval()
    try:
        total=0.0; positions=0
        for _ in range(num_batches):
            Xv,Yv,_=get_batch(val_tokens,batch_size,block_size,rng)
            lv=loss_fn(model,Xv,Yv); mx.eval(lv)
            n=Yv.shape[0]*Yv.shape[1]; total += lv.item()*n; positions += n
        return total/positions
    finally: model.train(was_training)

## 5. Gradient norm：数学与shape

把整棵梯度树所有叶子的元素展成一个向量g，全局L2范数为 `||g||₂ = sqrt(sum_i g_i²)`。若g=[3,4]，范数为5；这是手算例，不是模型梯度。

每片梯度叶子先逐元素平方并求和，再把这些标量相加、开平方。最后结果shape为()。它衡量梯度整体尺度，不直接等于Adam更新幅度，因为Adam还使用m/v、学习率和epsilon。

In [ ]:
def global_grad_norm(grad_tree):
    leaves=tree_flatten(grad_tree)
    squared_sums=[mx.sum(mx.square(leaf)) for _,leaf in leaves]
    if not squared_sums: raise ValueError('Gradient tree has no array leaves')
    return mx.sqrt(sum(squared_sums))
# Paper arithmetic only: sqrt(3**2 + 4**2) = 5.

## 6. Optional gradient clipping

Global norm clipping在optimizer update前限制梯度树的整体L2范数。阈值c>0时，scale=min(1,c/(G+epsilon_clip))，每片梯度叶子乘同一个scale。它不会保证loss下降，也不会修复坏数据、错误目标、NaN参数或不合适学习率；Adam仍会用自己的m/v处理裁剪后的梯度。

In [ ]:
def clip_grad_tree_global_norm(grad_tree,max_norm,eps_clip=1e-12):
    if not isinstance(max_norm,(int,float)) or max_norm<=0: raise ValueError('max_norm must be positive')
    norm=global_grad_norm(grad_tree)
    mx.eval(norm)
    scale=mx.minimum(mx.array(1.0,dtype=mx.float32),mx.array(max_norm,dtype=mx.float32)/(norm+eps_clip))
    clipped=tree_map(lambda g:g*scale,grad_tree)
    return clipped,norm,scale

## 7. NaN、Inf与发散检测

NaN是Not a Number，Inf是正/负无穷。Divergence（发散）可能表现为loss或gradient norm持续异常增大，但单个较大loss不自动证明发散。检查非有限值只能标记异常，不能自动诊断根因。

In [ ]:
def scalar_is_finite(value):
    mx.eval(value)
    return bool(mx.all(mx.isfinite(value)).item())
def assert_finite_training_values(loss,grad_norm):
    if not scalar_is_finite(loss): raise FloatingPointError('Training loss is NaN or Inf')
    if not scalar_is_finite(grad_norm): raise FloatingPointError('Gradient norm is NaN or Inf')

## 8. 初始化状态与稳定性单步骨架

Adam仍显式选择bias_correction=True（MLX v0.32.2默认False）。模型、梯度函数、optimizer和状态容器必须匹配。下面只定义单步骨架；可选clip发生在optimizer.update之前，非有限值检查也发生在更新之前。

In [ ]:
model_kwargs=make_model_kwargs(baseline_candidate,vocab_size)
mx.random.seed(seeds['model']); model=MiniGPT(**model_kwargs); model.train()
loss_and_grad_fn=nn.value_and_grad(model,loss_fn)
optimizer=optim.Adam(**make_optimizer_kwargs(baseline_candidate)); optimizer.init(model.trainable_parameters())
training_state=[model.state,optimizer.state]; mx.eval(training_state)
def stability_aware_step(X,Y,max_grad_norm=None):
    loss_before_update,grads=loss_and_grad_fn(model,X,Y)
    grad_norm=global_grad_norm(grads)
    mx.eval(loss_before_update,grad_norm)
    assert_finite_training_values(loss_before_update,grad_norm)
    if max_grad_norm is not None:
        grads,preclip_norm,scale=clip_grad_tree_global_norm(grads,max_grad_norm)
        mx.eval(preclip_norm,scale)
    optimizer.update(model,grads)
    mx.eval(training_state)
    return loss_before_update,grad_norm
# Definitions only. No training call is made in this master.

## 9. 本节边界与计数

本节只准备初始化、global gradient norm、optional clipping、NaN/Inf/divergence detection。没有执行训练循环或验证，没有填写loss、梯度或收敛结果；不提前展开资源控制或后续实验。

19.2已独立核心静态验收PASS，累计修正1/2。19.3初始生成0/2。后续按请求逐小节推进至大Step20全部材料完成后停止，不进入21。